# 07 · Error analysis do sistema escolhido

**Objetivo:** entender onde e por que o sistema escolhido perde NDCG@10 no validation set. O sistema escolhido é a hybrid search seguida do reranker sem o cross-encoder (registrado em [docs/EVALUATION.md](../docs/EVALUATION.md)). O NDCG@10 médio dele no validation set é um número só; este notebook o separa em partes para responder três perguntas:

1. **Em que etapa o sistema perde:** ao buscar os candidatos (retrieval) ou ao ordená-los (ranking)?
2. **Que tipo de erro ele comete:** o que os produtos mal colocados têm em comum, olhando todos os fields?
3. **Quanto da perda vem da própria avaliação:** de produtos sem `label` ou de `label` que os fields não explicam?

Nenhuma análise muda o sistema escolhido nem o resultado do test set. O que muda é o que podemos afirmar sobre o sistema e quais próximos passos fazem sentido.

**Regras deste notebook**
- **O test set não é usado.** Ele foi rodado uma única vez, no notebook 06, e todas as análises daqui são no validation set (97 queries).
- O sistema é o salvo em `models/` e carregado por `search.SearchPipeline.load`, o mesmo do notebook 05.
- As correções de typo (Etapa 3) e as categorias de erro (Etapa 5) foram feitas pelo assistente de IA, lendo as queries e os fields dos produtos, sem revisão humana. Elas ficam escritas no código, para que possam ser conferidas.

**Como ler este notebook**
- Cada etapa começa com quatro blocos:
  - **O que é analisado:** a pergunta.
  - **Por que:** o que a resposta permite decidir.
  - **Como:** a metodologia, passo a passo.
  - **Exemplo,** quando ajuda a entender.
- Cada seção pode ser lida sozinha. Toda tabela tem um título que diz o que é medido e em quais dados.
- As conclusões ficam logo abaixo da célula que as sustenta, sempre no mesmo formato:
  - **Resultados:** o que as saídas mostram, um fato por item.
  - **Impacto (nossa interpretação):** o que o resultado muda no relatório ou nos próximos passos. Não foi medido.
  - **Limitações:** o que a medição não captura.
  - **Escolhas e alternativas:** cada escolha, se está decidida ou só proposta, e as outras opções.
- A Etapa 7 resume todas as análises numa tabela: o que foi analisado, por que, o que foi encontrado e o impacto.

**Etapas**
1. Onde o sistema perde: no retrieval ou no ranking
2. Os produtos sem `label` no top 10
3. Queries com typo: a correção ajudaria?
4. Produtos quase idênticos recebem o mesmo `label`?
5. Que tipo de erro o sistema comete: categorias com leitura de todos os fields
6. O sistema põe os produtos mais populares no topo?
7. Síntese: o que foi encontrado e o impacto

## TL;DR

- **A perda está no ranking, não no retrieval.** Se os 100 candidatos da hybrid search fossem ordenados da melhor forma possível (o oracle), o NDCG@10 médio seria 0,933. O sistema chega a 0,776. Então o retrieval perde 0,067 e o ranking perde 0,156. O reranker já reduziu a perda do ranking de 0,186 para 0,156 (Etapa 1.1).
- **Em 39 das 49 queries com NDCG@10 < 0,9, a maior parte da perda está no ranking;** em 3, está no retrieval; em 7, o top 10 é quase todo de produtos sem `label` (Etapa 1.3).
- **Os produtos sem `label` no top 10 provavelmente são melhores do que a métrica supõe.** Entre os produtos julgados do top 10 com o mesmo `product_class` da `query_class`, só 0,9% são Irrelevant. Sem os produtos sem `label`, o NDCG@10 médio sobe de 0,776 para 0,831 (Etapa 2).
- **Não dá para medir o ganho de corrigir typos com estes `label`:** com a query corrigida, o sistema traz produtos que não foram julgados (Etapa 3).
- **Os `label` são consistentes na maior parte dos casos:** 5,6% dos pares de produtos quase idênticos têm `label` diferentes, contra 34,4% de todos os pares da mesma query, e nenhum par quase idêntico é Exact × Irrelevant (Etapa 4).
- **As categorias com causa visível nos fields são tipo de produto errado (21% da perda) e atributo não atendido (9%).** Por exemplo, na query "tufted chair with gold legs", 21 dos 28 Exact têm `legcolor` gold, e nenhum dos 4 produtos do top 10 que não são Exact tem (Etapa 5).
- **Em 15 queries (21% da perda), não encontramos em nenhum field algo que explique a diferença entre os Exact e os Partial.** Por exemplo, dois headboards com a mesma `product_description`, a mesma cor e quase os mesmos key:value pairs: um é Exact e o outro é Partial. Isso é uma limitação do dataset (Etapa 5.3).
- **O sistema não põe os produtos mais populares no topo.** O top 10 tem um pouco menos ratings que os 100 candidates (mediana de `log_rating_count` −0,404 [−0,644; −0,170]), e o reranker, apesar das features de rating, não muda a popularidade do top 10 em relação à hybrid search (−0,082 [−0,250; 0,101]). A fração de produtos sem rating no top 10 não se distingue da dos candidates. Os erros de ordem também não se concentram em produtos populares nem em produtos sem rating (Etapa 6.3).

## Termos usados no notebook

| Termo | Significado | Exemplo |
|---|---|---|
| query | Texto que uma pessoa digitou na busca da loja (coluna `query` de `query.csv`). | "tufted chair with gold legs" |
| catalog | Os 42.994 produtos de `product.csv`. | |
| label | Julgamento humano de um par (query, produto): Exact, Partial ou Irrelevant. Pelas [diretrizes do WANDS](https://github.com/wayfair/WANDS/blob/main/Product%20Search%20Relevance%20Annotation%20Guidelines.pdf), Exact atende a query inteira; Partial é o tipo de produto certo, mas sem algum modificador da query; Irrelevant não serve. | |
| judged pair | Par (query, produto) que tem `label`. Um produto retornado para uma query com que não forma judged pair é um produto **sem `label`** (`no label` no código). | |
| field | Uma das colunas de texto do produto: `product_name`, `product_class`, `category_hierarchy`, `product_description` e `product_features`. | |
| key:value pair | Cada item de `product_features`. Em `legcolor : gold`, a key é `legcolor` e o value é `gold`. | |
| `query_class` / `product_class` | Classe da query (em `query.csv`) e classe do produto (em `product.csv`). `product_class` pode ter mais de uma classe, separadas por `\|`. | `Accent Chairs` |
| retrieval | Primeira fase da busca: escolher, entre os 42.994 produtos, os candidatos. Aqui é a hybrid search, que combina BM25 e embeddings. | |
| candidates | Os 100 primeiros produtos do retrieval. São os únicos que o reranker reordena. | |
| ranking / reranker | Segunda fase: um modelo (LightGBM LambdaMART) dá uma nota a cada um dos 100 candidates e os reordena. | |
| sistema escolhido | Hybrid search + reranker sem o cross-encoder, salvo em `models/hybrid_reranker_no_cross_encoder.*`. | |
| oracle | O melhor top 10 possível usando só os 100 candidates: todos os Exact primeiro, depois os Partial, depois o resto. O NDCG@10 do oracle é o teto do reranker. | |
| retrieval loss | 1 − NDCG@10 do oracle. O quanto se perde porque os candidates não têm produtos bons o suficiente. | |
| ranking loss | NDCG@10 do oracle − NDCG@10 do sistema. O quanto se perde porque o reranker não põe os melhores candidates no topo. | |
| pooling | Como os `label` foram coletados: só os produtos que algum sistema da Wayfair retornou para a query foram julgados. | |
| pooling bias | Efeito do pooling: um sistema que traz produtos que nenhum sistema da Wayfair trouxe recebe nota 0 por eles, mesmo que sejam bons. | |
| error analysis | Olhar as queries com nota baixa para entender por que o sistema errou. | |
| typo | Erro de digitação na query. | `ligth bulb` |
| rating | As colunas `rating_count` (número de ratings de clientes), `average_rating` e `review_count` de `product.csv`. 22% dos produtos não têm rating. | |
| `log_rating_count` | log(1 + `rating_count`), com 0 para produto sem rating. É uma feature do reranker e, na Etapa 6, a medida de popularidade. | 2,4 ≈ 10 ratings |
| Spearman correlation | Correlação entre as posições de dois rankings, de −1 a 1. 0 = sem relação; só capta relações monótonas. | |
| par invertido | Par de produtos julgados da mesma query com `label` diferentes, em que o de `label` pior está acima. | Partial na posição 3, Exact na 15 |
| TF-IDF | Representação de um texto como um vetor com um peso por token: o peso cresce com quantas vezes o token aparece no texto e diminui com quantos produtos do catalog têm o token. | |
| cosine | Similaridade entre dois vetores TF-IDF, de 0 a 1. 1 = os mesmos tokens nas mesmas proporções. | |
| produtos quase idênticos | Dois produtos julgados para a mesma query cujo texto, somando os cinco fields, tem cosine ≥ 0,95. | |
| token | Palavra do texto, depois de remover acentos, passar para minúsculas e separar em tudo que não é letra ou dígito (`text.tokenize`). | `'Fawkes 36" Blue-Vanity'` → `fawkes`, `36`, `blue`, `vanity` |

### Métricas

As métricas seguem `docs/EVALUATION.md` e são calculadas por query; as tabelas mostram a média entre as queries.

| Métrica | O que mede | Queries |
|---|---|---|
| **NDCG@10** (principal) | Qualidade da ordem dos 10 primeiros. **Gain** é quanto cada produto vale: Exact = 2, Partial = 1, Irrelevant = 0, e produto sem `label` = 0. Os primeiros pesam mais. Vai de 0 a 1; 1 é a melhor ordem possível usando todos os produtos julgados da query. | todas |
| **condensed NDCG@10** | NDCG@10 depois de tirar da lista os produtos sem `label`. Mostra quanto do NDCG@10 depende de dar gain 0 a esses produtos. | todas |
| **oracle NDCG@10** | NDCG@10 do oracle (definido na tabela de termos): o máximo que o reranker pode alcançar com os 100 candidates. | todas |

## Setup

Carrega os dados e o sistema escolhido. O sistema lê os embeddings dos produtos do cache em `cache/`; sem o cache, eles são recalculados (cerca de meia hora numa CPU).

In [1]:
import platform
import time

import lightgbm
import numpy as np
import pandas as pd
import sklearn

from wands_search import data, error_analysis, evaluate, metrics, plots, ranking, search

plots.apply_style()
pd.set_option("display.max_colwidth", 120)

products = data.load_products()
queries = data.load_queries()
labels = data.deduplicate_labels(data.load_labels())
labels_by_query = metrics.labels_by_query(labels)
query_split = evaluate.load_split()
validation_queries = evaluate.split_queries(queries, query_split, "validation")
validation_labels = labels[labels["query_id"].isin(validation_queries["query_id"])]
products_by_id = products.set_index("product_id")
query_class_of = validation_queries.set_index("query_id")["query_class"]

start = time.perf_counter()
pipeline = search.SearchPipeline.load(products)
print(f"selected system loaded in {time.perf_counter() - start:.0f} s")
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| lightgbm", lightgbm.__version__)
print("validation queries:", len(validation_queries), "| judged pairs of the validation set:", len(validation_labels))

selected system loaded in 48 s
python 3.13.5 | pandas 3.1.1 | numpy 2.5.4 | scikit-learn 1.9.1 | lightgbm 4.7.0
validation queries: 97 | judged pairs of the validation set: 48321


## Etapa 1 · Onde o sistema perde: no retrieval ou no ranking

**O que é analisado:** o sistema tem duas etapas. O retrieval escolhe 100 candidates entre os 42.994 produtos, e o reranker ordena esses 100. Quando o NDCG@10 de uma query é baixo, a culpa pode ser de qualquer uma delas:
- **do retrieval,** se os produtos bons nem chegaram aos 100 candidates;
- **do ranking,** se estavam entre os candidates, mas o reranker não os pôs no top 10.

**Por que:** para saber qual etapa vale a pena melhorar. Se a perda está no retrieval, o próximo passo é buscar candidates melhores. Se está no ranking, é melhorar o reranker.

**Como:**
1. Para cada query do validation set, rodamos o sistema escolhido e guardamos os 100 candidates da hybrid search e a lista final do reranker.
2. Montamos o **oracle:** ordenamos os 100 candidates pelo `label`, com os Exact primeiro, depois os Partial, depois o resto. Isso é o melhor top 10 que qualquer reranker conseguiria com esses candidates.
3. Calculamos o NDCG@10 do oracle e o do sistema. Isso dá uma escada de três degraus:

| Degrau | O que significa |
|---|---|
| 1 (perfeito) | O top 10 tem os melhores produtos julgados da query, na ordem certa |
| oracle | O melhor top 10 possível com os 100 candidates |
| sistema | O top 10 que o reranker de fato monta com esses mesmos 100 candidates |

4. A queda de 1 até o oracle é a **retrieval loss**: produtos bons que não entraram nos candidates, e que nenhuma ordenação recupera. A queda do oracle até o sistema é a **ranking loss**: produtos bons que estavam nos candidates, mas ficaram fora do top 10. Por construção, as duas somam 1 − NDCG@10 do sistema em cada query.

**Exemplo (hipotético):** o retrieval traz 100 candidates para uma query, e 15 deles são Exact. Com os 15 no topo, o top 10 seria todo Exact: o oracle é alto, por exemplo 0,95. Se o reranker põe só 6 Exact no top 10, o sistema fica, por exemplo, em 0,75. Nessa query, o retrieval perde 0,05 e o ranking perde 0,20: a perda está no ranking.

A célula abaixo roda o sistema nas 97 queries do validation set e calcula, por query, o NDCG@10 do sistema, o da hybrid search sozinha, o do oracle e o condensed NDCG@10.

In [2]:
# Top 200 of the hybrid search (the candidates) and final list of the selected system, per validation query.
candidates_by_query, selected_by_query = {}, {}
for query_id, query_text in validation_queries[["query_id", "query"]].itertuples(index=False):
    candidates_by_query[query_id] = pipeline.retriever.search(query_text, search.MAX_RESULTS)["product_id"].tolist()
    selected_by_query[query_id] = pipeline.search(query_text, k=search.MAX_RESULTS).results["product_id"].tolist()

selected_rows = {}
for query_id, query_text, query_class in validation_queries[["query_id", "query", "query_class"]].itertuples(index=False):
    query_labels = labels_by_query.get(query_id, {})
    candidates = candidates_by_query[query_id]
    selected = selected_by_query[query_id]
    selected_rows[query_id] = {
        "query": query_text,
        "query_class": query_class,
        "ndcg_hybrid": metrics.ndcg_at_k(candidates, query_labels),
        "ndcg": metrics.ndcg_at_k(selected, query_labels),
        "oracle": metrics.oracle_ndcg_at_k(candidates[:ranking.CANDIDATES_K], query_labels),
        "condensed_ndcg": metrics.condensed_ndcg_at_k(selected, query_labels),
        "no_label_in_top10": sum(product_id not in query_labels for product_id in selected[:10]),
    }
selected_per_query = pd.DataFrame.from_dict(selected_rows, orient="index").rename_axis("query_id")
selected_per_query["retrieval_loss"] = 1 - selected_per_query["oracle"]
selected_per_query["ranking_loss"] = selected_per_query["oracle"] - selected_per_query["ndcg"]
selected_per_query["ranking_loss_hybrid"] = selected_per_query["oracle"] - selected_per_query["ndcg_hybrid"]
selected_per_query["main_loss"] = [
    error_analysis.main_loss_stage(row.ndcg, row.retrieval_loss, row.ranking_loss, row.no_label_in_top10)
    for row in selected_per_query.itertuples()
]
print("queries with an undefined NDCG@10:", int(selected_per_query["ndcg"].isna().sum()))

queries with an undefined NDCG@10: 0


### 1.1 · A escada em média: quanto o retrieval e o ranking perdem

A tabela mostra o NDCG@10 médio de cada degrau da escada da Etapa 1. A hybrid search sozinha entra como referência: o reranker reordena os mesmos 100 candidates dela, então o oracle é o mesmo para as duas.

In [3]:
ladder = pd.DataFrame({
    "mean_ndcg@10": [1.0, selected_per_query["oracle"].mean(), selected_per_query["ndcg"].mean(),
                     selected_per_query["ndcg_hybrid"].mean()],
}, index=pd.Index(["perfect order", "oracle: best order of the top 100 candidates",
                   "selected system: hybrid search + reranker", "hybrid search alone, no reranker"], name="level"))
display(plots.table(ladder, f"Mean NDCG@10 of each level, validation set ({len(selected_per_query)} queries)", precision=3,
                    note="The oracle sorts the top 100 candidates of the hybrid search by label (Exact, then Partial, then the rest). "
                         "The reranker reorders those same 100 candidates, so the oracle is the best the reranker can reach."))

losses = pd.DataFrame({
    "mean_loss": [selected_per_query["retrieval_loss"].mean(), selected_per_query["ranking_loss"].mean(),
                  selected_per_query["ranking_loss_hybrid"].mean()],
}, index=pd.Index(["retrieval loss = 1 - oracle", "ranking loss of the selected system = oracle - NDCG@10",
                   "ranking loss of the hybrid search alone = oracle - NDCG@10"], name="loss"))
display(plots.table(losses, "Mean NDCG@10 loss of each stage, validation set", precision=3,
                    note="Retrieval loss + ranking loss = 1 - NDCG@10 of the system, for each query."))

,mean_ndcg@10
level,
perfect order,1.000
oracle: best order of the top 100 candidates,0.933
selected system: hybrid search + reranker,0.776
"hybrid search alone, no reranker",0.747


,mean_loss
loss,
retrieval loss = 1 - oracle,0.067
ranking loss of the selected system = oracle - NDCG@10,0.156
ranking loss of the hybrid search alone = oracle - NDCG@10,0.186


**Resultados**
- **O oracle chega a 0,933,** então o retrieval perde só 0,067: os 100 candidates quase sempre têm produtos bons suficientes para um top 10 quase perfeito.
- **O sistema chega a 0,776,** então o ranking perde 0,156, mais que o dobro da retrieval loss.
- **O reranker reduziu a ranking loss de 0,186 para 0,156:** a hybrid search sozinha tem NDCG@10 de 0,747 com o mesmo oracle (0,933 − 0,747 = 0,186).

**Impacto (nossa interpretação)**
- A próxima melhoria deve ir para o ranking: os produtos certos já estão nos candidates, e o reranker é que não os põe no top 10. Aumentar o número de candidates ou trocar o retrieval mexe só nos 0,067.

**Limitações**
- O oracle conta só os produtos julgados. Um candidate sem `label` vale 0 no oracle, mesmo que seja bom (Etapa 2). Por isso, o oracle real pode ser maior que 0,933.
- O teto 1 usa os melhores produtos **julgados** da query, não do catalog inteiro.

### 1.2 · Um exemplo real: "tufted chair with gold legs"

Para ver a escada da Etapa 1 numa query, usamos a query 50, "tufted chair with gold legs" (`query_class` Accent Chairs). A tabela mostra o top 10 do sistema escolhido com o `label` de cada produto e, depois, o NDCG@10 do oracle e do sistema.

In [4]:
EXAMPLE_QUERY_ID = 50
example_labels = labels_by_query[EXAMPLE_QUERY_ID]
example_candidates = candidates_by_query[EXAMPLE_QUERY_ID][:ranking.CANDIDATES_K]
example_top10 = pd.DataFrame({
    "rank": range(1, 11),
    "product_name": products_by_id.loc[selected_by_query[EXAMPLE_QUERY_ID][:10], "product_name"].to_numpy(),
    "label": [example_labels.get(product_id, ranking.NO_LABEL) for product_id in selected_by_query[EXAMPLE_QUERY_ID][:10]],
}).set_index("rank")
print(f"query {EXAMPLE_QUERY_ID}: {selected_per_query.at[EXAMPLE_QUERY_ID, 'query']!r}")
print("Exact among the top 100 candidates:", sum(example_labels.get(product_id) == "Exact" for product_id in example_candidates),
      "| Exact of the query:", sum(label == "Exact" for label in example_labels.values()))
display(plots.table(example_top10, f"Top 10 of the selected system for query {EXAMPLE_QUERY_ID}, with the label of each product"))
display(plots.table(selected_per_query.loc[[EXAMPLE_QUERY_ID], ["oracle", "ndcg", "retrieval_loss", "ranking_loss"]],
                    f"NDCG@10 of the oracle and of the selected system for query {EXAMPLE_QUERY_ID}", precision=3))

query 50: 'tufted chair with gold legs'
Exact among the top 100 candidates: 14 | Exact of the query: 28


,product_name,label
rank,,
1,danis tufted upholstered side chair in antique gold,Partial
2,ivo 30 '' wide tufted wingback chair,Partial
3,aveline tufted polyester wingback chair,Exact
4,asimina 28.3 '' wide tufted wingback chair,Partial
5,esmund 20 '' wide tufted velvet side chair,Exact
6,31.5 '' wide tufted velvet wingback chair,Exact
7,claudie 26 '' wide tufted polyester side chair,Exact
8,"modern soft velvet material ergonomics tufted armless chair upholstered accent chair , living room chair bedroom chai...",Partial
9,ilene 32 '' wide tufted velvet barrel chair,Exact


,oracle,ndcg,retrieval_loss,ranking_loss
query_id,,,,
50,1.000,0.738,0.000,0.262


**Resultados**
- **14 dos 28 Exact da query estão nos 100 candidates.** Com 10 deles no topo, o oracle tem NDCG@10 = 1: a retrieval loss é 0.
- **O sistema põe 6 Exact e 4 Partial no top 10,** e o NDCG@10 fica em 0,738: toda a perda (0,262) é ranking loss.
- **Todos os produtos do top 10 são tufted chairs.** O que separa os Exact dos Partial é o atributo "gold legs", e a Etapa 5.2 mostra que ele está em `product_features` (key `legcolor`).

**Limitações**
- É uma única query, escolhida como exemplo. A distribuição das 97 queries está na Etapa 1.3.

### 1.3 · Em quantas queries a perda está no retrieval e em quantas no ranking

**Como:** cada query do validation set recebe a etapa onde está a maior parte da perda, nesta ordem (`error_analysis.main_loss_stage`):
1. **ok:** NDCG@10 ≥ 0,9. A query fica fora da error analysis.
2. **evaluation:** 5 ou mais dos 10 primeiros produtos não têm `label`. Aqui a nota baixa diz mais sobre o pooling do que sobre o sistema.
3. **retrieval:** a retrieval loss é maior que a ranking loss.
4. **ranking:** nos outros casos.

In [5]:
stage_summary = selected_per_query.groupby("main_loss").agg(
    queries=("ndcg", "size"), mean_ndcg=("ndcg", "mean"), mean_retrieval_loss=("retrieval_loss", "mean"),
    mean_ranking_loss=("ranking_loss", "mean"), mean_no_label_in_top10=("no_label_in_top10", "mean"),
).sort_values("queries", ascending=False)
plots.table(stage_summary, "Validation queries by the stage with the larger NDCG@10 loss", precision=3,
            note="ok: NDCG@10 >= 0.9. evaluation: 5 or more of the top 10 have no label. Otherwise retrieval when "
                 "1 - oracle > oracle - NDCG@10, and ranking in the other cases.")

,queries,mean_ndcg,mean_retrieval_loss,mean_ranking_loss,mean_no_label_in_top10
main_loss,,,,,
ok (NDCG@10 >= 0.9),48,0.981,0.000,0.019,0.021
ranking,39,0.650,0.046,0.304,0.846
evaluation (top 10 mostly no label),7,0.127,0.574,0.299,8.857
retrieval,3,0.663,0.240,0.097,0.667


**Resultados**
- **48 das 97 queries têm NDCG@10 ≥ 0,9** (média 0,981). As outras 49 são as queries analisadas nas próximas etapas.
- **A perda está principalmente no ranking em 39 das 49 queries.** Nelas, a ranking loss média é 0,304, e a retrieval loss média, 0,046.
- **Em 7 queries, o top 10 é quase todo de produtos sem `label`** (média de 8,9 por query) e o NDCG@10 médio é 0,127.
- **A perda está principalmente no retrieval em só 3 queries.**

**Impacto (nossa interpretação)**
- Confirma a Etapa 1.1 query a query: o problema típico não é faltar produto bom nos candidates, e sim ordená-los. As 7 queries de "evaluation" não dizem muito sobre o sistema, porque a nota delas depende de produtos que ninguém julgou (Etapa 2).

**Limitações**
- O limite de 0,9 e o de 5 produtos sem `label` são escolhas nossas. Com outros limites, o número de queries em cada grupo muda, mas a ranking loss média continua maior que a retrieval loss (Etapa 1.1).

**Escolhas e alternativas**

| Escolha | Status | Alternativas |
|---|---|---|
| Analisar as queries com NDCG@10 < 0,9 | decidida | as N piores queries; todas as queries, ponderadas pela perda |
| Oracle sobre os 100 candidates | decidida | oracle sobre os 200 da hybrid search (mede o retrieval com outra profundidade) |

## Etapa 2 · Os produtos sem `label` no top 10

**O que é analisado:** os produtos que o sistema põe no top 10, mas que não têm `label` para a query.

**Por que:** a métrica dá gain 0 a todo produto sem `label`, como se fosse Irrelevant. Os `label` do WANDS vêm de pooling: só foram julgados os produtos que algum sistema da Wayfair retornou. Se o nosso sistema traz produtos bons que os sistemas da Wayfair não trouxeram, o NDCG@10 subestima o nosso sistema (pooling bias). Precisamos saber se isso acontece para interpretar a nota.

**Como:** não dá para saber o `label` que um produto sem `label` teria. Mas dá para usar uma pista medida nos produtos que têm `label`:
1. Para cada produto do top 10, olhamos se o `product_class` dele tem alguma classe em comum com a `query_class` da query (`error_analysis.shares_product_class`). Isso divide os produtos em "mesma classe" e "outra classe".
2. Entre os produtos **com** `label`, medimos quantos são Irrelevant em cada grupo.
3. Se quase nenhum produto julgado da mesma classe é Irrelevant, é razoável esperar o mesmo dos sem `label` da mesma classe.
4. Por fim, comparamos o NDCG@10 com o condensed NDCG@10, que tira os produtos sem `label` da lista antes de cortar no top 10.

**Exemplo:** para a query "pastel pink desk chair" (`query_class` Office Chairs), uma cadeira de escritório rosa sem `label` é da mesma classe; um quadro rosa é de outra classe.

### 2.1 · O `label` dos produtos do top 10, por classe

As tabelas usam os 970 produtos do top 10 das 97 queries (10 por query). A primeira conta os produtos de cada grupo. A segunda mostra a distribuição dos `label`, incluindo "no label". A terceira mostra a mesma distribuição só entre os produtos julgados.

In [6]:
top10_rows = []
for query_id, selected in selected_by_query.items():
    query_labels = labels_by_query.get(query_id, {})
    for product_id in selected[:10]:
        top10_rows.append({
            "query_id": query_id,
            "product_id": product_id,
            "label": query_labels.get(product_id, ranking.NO_LABEL),
            "same_class": error_analysis.shares_product_class(query_class_of[query_id], products_by_id.at[product_id, "product_class"]),
        })
selected_top10 = pd.DataFrame(top10_rows)
selected_top10["class_match"] = selected_top10["same_class"].map({True: "same class as query_class", False: "other class"})

class_counts = selected_top10.groupby("class_match").size().rename("products")
display(plots.table(class_counts.to_frame(), "Products in the top 10 of the selected system by class match, validation set",
                    precision=0, note="same class: the product_class of the product shares a class with the query_class of the query. "
                                      "Queries without query_class count as other class."))
label_shares = (pd.crosstab(selected_top10["class_match"], selected_top10["label"], normalize="index") * 100)[
    ["Exact", "Partial", "Irrelevant", ranking.NO_LABEL]]
display(plots.table(label_shares, "Label of the products in the top 10 of the selected system (% of each row), validation set",
                    note="no label: the pair (query, product) was not judged."))
judged_shares = (pd.crosstab(selected_top10.loc[selected_top10["label"] != ranking.NO_LABEL, "class_match"],
                             selected_top10.loc[selected_top10["label"] != ranking.NO_LABEL, "label"], normalize="index") * 100)[
    ["Exact", "Partial", "Irrelevant"]]
plots.table(judged_shares, "Label of the judged products in the top 10 of the selected system (% of each row), validation set",
            note="Only products with a label for the query.")

,products
class_match,
other class,350
same class as query_class,620


label,Exact,Partial,Irrelevant,no label
class_match,,,,
other class,36.3,36.3,13.7,13.7
same class as query_class,44.4,46.8,0.8,8.1


label,Exact,Partial,Irrelevant
class_match,,,
other class,42.1,42.1,15.9
same class as query_class,48.2,50.9,0.9


**Resultados**
- **620 dos 970 produtos do top 10 são da mesma classe da `query_class`;** 350 são de outra classe. A conta de "outra classe" inclui as queries sem `query_class`.
- **Entre os produtos julgados da mesma classe, só 0,9% são Irrelevant;** 48,2% são Exact e 50,9% Partial.
- **Entre os produtos julgados de outra classe, 15,9% são Irrelevant.**
- **8,1% dos produtos da mesma classe e 13,7% dos de outra classe não têm `label`.**

**Impacto (nossa interpretação)**
- Se os produtos sem `label` da mesma classe se comportam como os julgados da mesma classe, quase todos seriam Exact ou Partial, e não Irrelevant. Nesse caso, a métrica os conta como 0 quando valeriam 1 ou 2.

**Limitações**
- É uma inferência a partir dos produtos julgados. Não sabemos o `label` real dos produtos sem `label`, e eles podem ser diferentes dos julgados justamente porque nenhum sistema da Wayfair os trouxe.
- A classe é uma pista grosseira: um produto da mesma classe pode não ter o atributo pedido (cor, tamanho).

### 2.2 · Quanto os produtos sem `label` pesam no NDCG@10

A primeira tabela conta os produtos sem `label` no top 10, em média por query, separados pela classe como na Etapa 2.1. A segunda compara o NDCG@10 (produtos sem `label` com gain 0) com o condensed NDCG@10 (produtos sem `label` removidos).

In [7]:
no_label_per_query = (selected_top10[selected_top10["label"] == ranking.NO_LABEL]
                      .groupby(["query_id", "class_match"]).size().unstack(fill_value=0)
                      .reindex(selected_per_query.index, fill_value=0))
no_label_summary = pd.DataFrame({
    "mean_per_query": [no_label_per_query.sum(axis=1).mean(), no_label_per_query["same class as query_class"].mean(),
                       no_label_per_query["other class"].mean()],
}, index=pd.Index(["all products without label", "same class as query_class", "other class"], name="products without label in the top 10"))
display(plots.table(no_label_summary, "Products without label in the top 10 of the selected system, mean per validation query", precision=2))
print("queries with at least one product without label in the top 10:", int((no_label_per_query.sum(axis=1) > 0).sum()),
      "of", len(selected_per_query))
ndcg_versions = selected_per_query[["ndcg", "condensed_ndcg"]].mean().rename("mean").to_frame()
ndcg_versions.index = ["NDCG@10 (products without label have gain 0)", "condensed NDCG@10 (products without label removed)"]
plots.table(ndcg_versions, "Mean NDCG@10 of the selected system with and without products without label, validation set", precision=3)

,mean_per_query
products without label in the top 10,
all products without label,1.01
same class as query_class,0.52
other class,0.49


queries with at least one product without label in the top 10: 31 of 97


,mean
NDCG@10 (products without label have gain 0),0.776
condensed NDCG@10 (products without label removed),0.831


**Resultados**
- **Em média, o top 10 tem 1,01 produto sem `label` por query:** 0,52 da mesma classe da `query_class` e 0,49 de outra classe.
- **Eles se concentram em poucas queries:** só 31 das 97 queries têm pelo menos um produto sem `label` no top 10.
- **Sem os produtos sem `label`, o NDCG@10 médio sobe de 0,776 para 0,831.**

**Impacto (nossa interpretação)**
- O NDCG@10 que reportamos (0,776) é provavelmente um piso, e o condensed NDCG@10 (0,831) é um teto otimista. O relatório deve mostrar os dois números.
- Nas queries com muitos produtos sem `label` (as 7 de "evaluation" da Etapa 1.3), a nota baixa não deve ser lida como erro do sistema sem olhar os produtos.

**Limitações**
- O condensed NDCG@10 não é o valor real: ele supõe que a ordem dos produtos julgados não muda quando os sem `label` são removidos, e ignora o valor que eles teriam.

## Etapa 3 · Queries com typo: a correção ajudaria?

**O que é analisado:** as queries do validation set que têm erro de digitação, como "ligth bulb".

**Por que:** a lexical search procura os tokens da query no texto dos produtos. Um token com typo, como `minnestrista`, não aparece em nenhum produto. Se corrigir a query melhora muito a nota, um corretor ortográfico é um próximo passo que vale a pena.

**Como:**
1. O assistente de IA leu as 97 queries do validation set e encontrou 5 com typo. A correção é do assistente e fica no dicionário `TYPO_FIXES`.
2. Rodamos o sistema escolhido duas vezes para cada query: com o texto original e com o texto corrigido.
3. Comparamos o NDCG@10 e o condensed NDCG@10 das duas versões, usando **os mesmos `label`**, que são os da query original.

**Um detalhe importante:** pelas [diretrizes do WANDS](https://github.com/wayfair/WANDS/blob/main/Product%20Search%20Relevance%20Annotation%20Guidelines.pdf), quem julgou ignorou os typos e julgou pelo sentido da query. Então os `label` já valem para a query corrigida. Mas o pooling foi feito com a query original: só foram julgados os produtos que os sistemas da Wayfair trouxeram **para o texto com typo**.

### 3.1 · NDCG@10 com a query original e com a corrigida

In [8]:
# Validation queries with a typo, found by reading the queries; the correction is ours.
TYPO_FIXES = {
    383: "minnetrista",
    219: "trinsic towel rod",
    222: "small living room tables",
    224: "twin over full bunk beds cool designs",
    305: "light bulb",
}
assert set(TYPO_FIXES) <= set(validation_queries["query_id"])
typo_rows = []
for query_id, corrected_text in TYPO_FIXES.items():
    query_labels = labels_by_query.get(query_id, {})
    for version, query_text in (("original", selected_per_query.at[query_id, "query"]), ("corrected", corrected_text)):
        selected = pipeline.search(query_text, k=search.MAX_RESULTS).results["product_id"].tolist()
        typo_rows.append({
            "query_id": query_id, "version": version, "query": query_text,
            "ndcg": metrics.ndcg_at_k(selected, query_labels),
            "condensed_ndcg": metrics.condensed_ndcg_at_k(selected, query_labels),
            "no_label_in_top10": sum(product_id not in query_labels for product_id in selected[:10]),
        })
typo_results = pd.DataFrame(typo_rows).set_index(["query_id", "version"])
display(plots.table(typo_results[["query", "ndcg", "condensed_ndcg"]], "NDCG@10 of the selected system for the validation queries with a typo, "
                    "original and corrected query", precision=3,
                    note="The labels are the same in both versions: they were collected for the original query."))
plots.table(typo_results[["no_label_in_top10"]], "Products without label in the top 10, original and corrected query", precision=0)

**Resultados**
- **Em 2 das 5 queries ("ligth bulb" e "trinaic towel rod"), a versão original já tem NDCG@10 = 1.** Nas outras 3, ele fica abaixo de 0,9.
- **"minnestrista" fica com NDCG@10 = 0 nas duas versões,** com os 10 primeiros produtos sem `label` nas duas.
- **"trinsic towel rod" (corrigida) cai de 1 para 0,130:** 8 dos 10 primeiros produtos não têm `label`. O condensed NDCG@10 continua 1.
- **"small living room tables" (corrigida) cai de 0,064 para 0:** 9 dos 10 primeiros produtos originais e todos os 10 corrigidos não têm `label`. O condensed NDCG@10 sobe de 0,516 para 0,691.
- **"twin over full bunk beds cool designs" (corrigida) sobe de 0,450 para 0,605.**

**Impacto (nossa interpretação)**
- A correção muda quais produtos o sistema traz, e os produtos novos quase nunca foram julgados. Por isso, o NDCG@10 da versão corrigida não mede o ganho da correção: ele pode até cair, como em "trinsic towel rod".
- As 3 queries com typo abaixo de 0,9 respondem por 11,5% da perda total (Etapa 5.5). Um corretor ortográfico é um próximo passo promissor, mas o relatório não pode prometer um número para ele com estes `label`.

**Limitações**
- São 5 queries, encontradas lendo o validation set; não há uma lista oficial de typos no WANDS.
- A correção foi escolhida pelo assistente e pode não ser a que a pessoa queria (por exemplo, `minnetrista`).

### 3.2 · Exemplo: "small loving roomtables"

O top 10 do sistema escolhido para a query 222, com o texto original e com o texto corrigido ("small living room tables"). O `label` de cada produto está entre parênteses.

In [9]:
TYPO_EXAMPLE_QUERY_ID = 222
typo_example = {}
for version, query_text in (("original", selected_per_query.at[TYPO_EXAMPLE_QUERY_ID, "query"]), ("corrected", TYPO_FIXES[TYPO_EXAMPLE_QUERY_ID])):
    selected = pipeline.search(query_text, k=10).results
    typo_example[version] = [f"{name} ({labels_by_query[TYPO_EXAMPLE_QUERY_ID].get(product_id, ranking.NO_LABEL)})"
                             for product_id, name in zip(selected["product_id"], selected["product_name"])]
plots.table(pd.DataFrame(typo_example, index=pd.RangeIndex(1, 11, name="rank")),
            f"Top 10 of the selected system for query {TYPO_EXAMPLE_QUERY_ID}, original and corrected, with the label in parentheses")

,original,corrected
rank,,
1,pringle chic small wooden console table (no label),pringle chic small wooden console table (no label)
2,caire frame nesting tables (no label),nesting small end table (no label)
3,mabon end table with storage (no label),"3-tier end table , small book shelf , accent side , and end table (no label)"
4,alluna end table with storage (no label),addyson living room 3 piece coffee table set (no label)
5,borkur 2 piece coffee table set (no label),ellert 3 piece nesting tables (no label)
6,ellert 3 piece nesting tables (no label),caire frame nesting tables (no label)
7,kenton petite end table with storage (no label),borkur 2 piece coffee table set (no label)
8,saroyan 2 piece coffee table set (no label),wiley frame 2 nesting tables (no label)
9,littlehampt extendable floor shelf coffee table with storage (no label),alluna end table with storage (no label)


**Resultados**
- **Com o texto original, só 1 dos 10 primeiros produtos tem `label`** ("vikki c table nesting tables", Exact, na posição 10).
- **Com o texto corrigido, nenhum dos 10 tem `label`.** Os produtos são mesas pequenas (console tables, nesting tables, end tables, coffee table sets), e a posição 10 é um vaso.

**Impacto (nossa interpretação)**
- Os dois tops parecem razoáveis para quem buscou "small living room tables", mas a métrica não consegue mostrar isso porque quase nenhum desses produtos foi julgado. É o caso mais claro de pooling bias do validation set.

## Etapa 4 · Produtos quase idênticos recebem o mesmo `label`?

**O que é analisado:** pares de produtos julgados para a mesma query cujo texto é quase igual, e se eles recebem o mesmo `label`.

**Por que:** antes de concluir que o sistema errou, precisamos saber se os `label` são confiáveis. Se dois produtos quase iguais recebem `label` diferentes com frequência, parte da perda do sistema vem dos `label`, e nenhuma ordenação a evitaria. Se recebem quase sempre o mesmo `label`, as métricas são confiáveis.

**Como:**
1. Juntamos o texto dos cinco fields de cada produto (`error_analysis.full_text`).
2. Transformamos cada texto num vetor TF-IDF, com os tokens de `text.tokenize` e o IDF calculado no catalog inteiro.
3. Para cada query do validation set, comparamos todos os pares de produtos julgados para ela pelo cosine dos vetores.
4. Contamos quantos pares têm `label` diferentes em cada faixa de cosine (≥ 0,80, ≥ 0,90, ≥ 0,95 e ≥ 0,99).
5. Como referência, contamos o mesmo para **todos** os pares de produtos julgados da mesma query, quase idênticos ou não.

**Exemplo:** "nautical upholstered panel headboard" e "beach upholstered panel headboard" têm a mesma `product_description` e quase os mesmos key:value pairs, então o cosine deles é próximo de 1.

In [10]:
NEAR_DUPLICATE_THRESHOLDS = (0.80, 0.90, 0.95, 0.99)
start = time.perf_counter()
product_texts = error_analysis.full_text(products)
near_pairs = error_analysis.near_duplicate_pairs(product_texts, validation_labels, threshold=min(NEAR_DUPLICATE_THRESHOLDS))
print(f"TF-IDF over the catalog and pairs of the validation set in {time.perf_counter() - start:.0f} s")

different_pairs, all_pairs = np.sum([error_analysis.label_disagreement_rate(group["label"].tolist())
                                     for _, group in validation_labels.groupby("query_id")], axis=0)
near_rows = {"all judged pairs of the same query": {
    "pairs": all_pairs, "queries": validation_labels["query_id"].nunique(),
    "different_labels_pct": different_pairs / all_pairs * 100,
    "exact_vs_irrelevant": sum((group["label"] == "Exact").sum() * (group["label"] == "Irrelevant").sum()
                               for _, group in validation_labels.groupby("query_id"))}}
for threshold in NEAR_DUPLICATE_THRESHOLDS:
    subset = near_pairs[near_pairs["cosine"] >= threshold]
    exact_irrelevant = ((subset["label_a"] + subset["label_b"]).isin(["ExactIrrelevant", "IrrelevantExact"])).sum()
    near_rows[f"cosine >= {threshold}"] = {"pairs": len(subset), "queries": subset["query_id"].nunique(),
                                          "different_labels_pct": subset["different"].mean() * 100,
                                          "exact_vs_irrelevant": exact_irrelevant}
near_summary = pd.DataFrame.from_dict(near_rows, orient="index").rename_axis("pairs of judged products")
display(plots.table(near_summary[["different_labels_pct"]], "Pairs of products judged for the same validation query with different labels (%)",
                    note="cosine: TF-IDF similarity of the five text fields of the two products; 1 means the same tokens in the same "
                         "proportions. The first row counts every pair, near-identical or not."))
near_counts = near_summary[["pairs", "queries", "exact_vs_irrelevant"]].astype(int)
display(plots.table(near_counts, "Number of pairs, of queries and of Exact vs Irrelevant pairs in each row", precision=0,
                    note="queries: validation queries with at least one pair in the row. exact_vs_irrelevant: pairs with Exact on one side "
                         "and Irrelevant on the other."))
near_095 = near_pairs[near_pairs["cosine"] >= 0.95]
print("queries with at least one near-identical pair (cosine >= 0.95) with different labels:",
      near_095.loc[near_095["different"], "query_id"].nunique(), "of", near_095["query_id"].nunique())

TF-IDF over the catalog and pairs of the validation set in 26 s


,different_labels_pct
pairs of judged products,
all judged pairs of the same query,34.4
cosine >= 0.8,10.7
cosine >= 0.9,7.1
cosine >= 0.95,5.6
cosine >= 0.99,6.5


,pairs,queries,exact_vs_irrelevant
pairs of judged products,,,
all judged pairs of the same query,"37,365,250",97,"1,783,643"
cosine >= 0.8,"56,957",95,90
cosine >= 0.9,"21,019",94,2
cosine >= 0.95,"12,684",88,0
cosine >= 0.99,168,49,0


queries with at least one near-identical pair (cosine >= 0.95) with different labels: 38 of 88


**Resultados**
- **Entre todos os pares de produtos julgados da mesma query, 34,4% têm `label` diferentes.** São 37.365.250 pares nas 97 queries, e 1.783.643 deles são Exact × Irrelevant.
- **Entre os pares quase idênticos (cosine ≥ 0,95), só 5,6% têm `label` diferentes.** São 12.684 pares em 88 queries.
- **Nenhum par quase idêntico é Exact × Irrelevant** (cosine ≥ 0,95). Com cosine ≥ 0,80, são 90 pares.
- **38 das 88 queries com pares quase idênticos têm pelo menos um par com `label` diferentes.**

**Impacto (nossa interpretação)**
- Quanto mais parecidos dois produtos são, mais vezes eles têm o mesmo `label`. Os `label` são consistentes na maior parte dos casos, e as métricas são confiáveis.
- Entre os pares quase idênticos (cosine ≥ 0,95), as discordâncias são entre Exact e Partial ou entre Partial e Irrelevant, nunca entre os extremos. A Etapa 5.3 mostra um exemplo.

**Limitações**
- O cosine de TF-IDF olha o texto. Dois produtos com texto quase igual podem diferir no que não está no texto (por exemplo, a imagem, que quem julgou via, segundo as diretrizes do WANDS). Não medimos isso.
- Com cosine ≥ 0,99, a taxa (6,5%) é um pouco maior que com ≥ 0,95, mas são só 168 pares.

**Escolhas e alternativas**

| Escolha | Status | Alternativas |
|---|---|---|
| Cosine de TF-IDF sobre os cinco fields, com limite de 0,95 | decidida | cosine dos embeddings do sistema; comparar só `product_name` e `product_features`; outros limites (a tabela mostra quatro) |

## Etapa 5 · Que tipo de erro o sistema comete

**O que é analisado:** as 49 queries do validation set com NDCG@10 < 0,9. Para cada uma, procuramos o motivo pelo qual o sistema pôs produtos que não são Exact no top 10, ou deixou Exact fora dele.

**Por que:** a Etapa 1 mostra que a perda está no ranking, mas não diz o que mudar no ranking. Agrupar os erros por causa transforma "o ranking perde" em causas concretas, e cada causa sugere uma ação diferente. Também separa os erros que o sistema pode corrigir dos que dependem da avaliação ou dos dados.

**Como:**
1. Para cada uma das 49 queries, listamos dois grupos de produtos:
   - os produtos do top 10 que **não** são Exact (Partial, Irrelevant ou sem `label`);
   - até 3 Exact que ficaram **fora** do top 10.
2. O assistente lê **todos os fields** desses produtos: `product_name`, `product_class`, `category_hierarchy`, `product_description` e `product_features`. Uma primeira leitura, só com `product_name` e `product_class`, mudou de categoria em 12 queries quando ele passou a ler todos os fields, então todos os fields são lidos.
3. Damos **uma** categoria a cada query: a que melhor descreve os produtos que causam a maior parte da perda.
4. Quando os fields permitem, medimos a categoria com uma contagem: por exemplo, quantos Exact e quantos Partial têm o atributo pedido pela query (Etapas 5.2 a 5.4).

As categorias:

| Categoria | Critério | Exemplo |
|---|---|---|
| `wrong_product_type` | Os produtos do top 10 que não são Exact são de outro tipo de produto, diferente do que a query pede. | "beds that have leds": as posições 6, 8, 9 e 10 são lâmpadas LED (`product_class` Light Bulbs), Irrelevant |
| `attribute_not_met` | O tipo de produto está certo, mas os fields mostram que os produtos do top 10 que não são Exact não têm o atributo pedido. | "tufted chair with gold legs": cadeiras sem `legcolor` gold |
| `compound_query` | A query pede dois tipos de produto ao mesmo tempo, e o sistema traz só um. | "entertainment stand end table" |
| `typo` | A query tem um erro de digitação (Etapa 3). | "minnestrista" |
| `proper_name` | A query depende de um nome de marca, coleção ou personagem. | "orren ellis l shaped desk" |
| `vague_query` | A query não diz que tipo de produto quer. | "living room ideas" |
| `pooling_bias` | A perda vem dos produtos sem `label` no top 10. Sem eles, o condensed NDCG@10 é alto. | "pastel pink desk chair": condensed NDCG@10 = 1 |
| `unexplained_labels` | Os produtos do top 10 que não são Exact têm o tipo e o atributo pedidos, segundo os fields, e mesmo assim são Partial. Não encontramos em nenhum field algo que explique a diferença em relação aos Exact. | "beach blue headboard" (Etapa 5.3) |

A célula abaixo registra a categoria de cada query e confere que as 49 queries com NDCG@10 < 0,9 têm uma categoria e só uma.

In [11]:
# Manual categories of the validation queries with NDCG@10 < 0.9 (one primary category each), after reading
# every field of the products in the top 10 that are not Exact and of the Exact products outside the top 10.
ERROR_CATEGORIES = {
    "wrong_product_type": [84, 88, 202, 454, 101, 277, 451, 14, 251, 334, 269],
    "attribute_not_met": [292, 22, 50, 478, 447, 36],
    "compound_query": [95, 93],
    "typo": [383, 222, 224],
    "proper_name": [223, 481, 137, 233, 458],
    "vague_query": [419, 449, 480],
    "pooling_bias": [304, 400, 375, 209],
    "unexplained_labels": [291, 151, 82, 466, 42, 425, 226, 275, 232, 382, 46, 76, 33, 402, 75],
}
category_of = {query_id: category for category, query_ids in ERROR_CATEGORIES.items() for query_id in query_ids}
below_threshold = set(selected_per_query.index[selected_per_query["ndcg"] < error_analysis.OK_NDCG])
assert set(category_of) == below_threshold, set(category_of) ^ below_threshold
selected_per_query["category"] = pd.Series(category_of).reindex(selected_per_query.index).fillna(error_analysis.STAGE_OK)
print("queries with NDCG@10 < 0.9:", len(below_threshold), "of", len(selected_per_query))
categorized = selected_per_query.loc[list(category_of), ["category", "query", "query_class", "ndcg", "oracle", "no_label_in_top10"]]
plots.table(categorized, "Validation queries with NDCG@10 < 0.9 and their error category", precision=2,
            note="oracle: best NDCG@10 that an ordering of the top 100 candidates can reach.")

queries with NDCG@10 < 0.9: 49 of 97


,category,query,query_class,ndcg,oracle,no_label_in_top10
query_id,,,,,,
84,wrong_product_type,full metal bed rose gold,Beds,0.09,0.82,1
88,wrong_product_type,rose gold lounge,Indoor Chaise Lounges,0.28,1.00,0
202,wrong_product_type,full mattress padded liner,Mattress Toppers and Pads,0.50,1.00,4
454,wrong_product_type,carpet 5x6,Area Rugs,0.54,1.00,1
101,wrong_product_type,48 in entry table with side by side drawer,Sofa & Console Tables,0.55,0.78,2
277,wrong_product_type,barstool patio sets,Patio Bar Stools,0.58,1.00,2
451,wrong_product_type,aiden black velvet bed,Beds,0.69,1.00,1
14,wrong_product_type,beds that have leds,Beds,0.72,1.00,0
251,wrong_product_type,coffee container,Canisters & Jars,0.86,1.00,0


**Resultados**
- **As 49 queries com NDCG@10 < 0,9 receberam uma categoria cada.** A maior é `unexplained_labels` (15 queries), seguida de `wrong_product_type` (11) e `attribute_not_met` (6).
- **7 queries têm de 7 a 10 produtos sem `label` no top 10:** 2 de `typo` (383 e 222), as 3 de `vague_query` e 2 de `pooling_bias` (304 e 400). São as 7 que a Etapa 1.3 marca como "evaluation".

**Limitações**
- As categorias vêm de uma única leitura, feita pelo assistente de IA, sem revisão humana. Algumas queries cabem em duas categorias; por exemplo, "donaldson teak couch" tem um nome próprio ("donaldson", que não aparece em nenhum produto do catalog), mas a perda vem de uma coffee table e de uma console table no top 10, então ficou em `wrong_product_type`.
- São 49 queries. As proporções por categoria (Etapa 5.5) mudariam com outra amostra.

### 5.2 · O atributo pedido está em `product_features`?

**O que é analisado:** se o atributo que a query pede (cor da perna, rodinhas, quantidade de banquetas) aparece nos key:value pairs de `product_features`, e se ele separa os Exact dos Partial.

**Por que:** é a forma de confirmar com números a diferença entre `attribute_not_met` e `unexplained_labels`. Em `attribute_not_met`, o atributo está nos Exact e falta nos produtos mal colocados: o sistema poderia usá-lo. Em `unexplained_labels`, os produtos mal colocados também têm o atributo, e os fields não explicam por que são Partial.

**Como:**
1. Para cada query, escolhemos as keys de `product_features` que descrevem o atributo e uma regra para o value (por exemplo, `legcolor` contém "gold"). As regras ficam em `ATTRIBUTE_CHECKS`.
2. Contamos quantos produtos têm o atributo em três grupos:
   - todos os Exact da query;
   - todos os Partial da query;
   - os produtos do top 10 do sistema que não são Exact.
3. Escolhemos 3 queries com causa visível (duas de `attribute_not_met` e uma de `wrong_product_type`) e 3 de `unexplained_labels`.

**Exemplo:** em "laundry basket with wheels", a key `wheelsorcastersincluded` diz se o cesto tem rodinhas.

In [12]:
def number_at_least(minimum):
    def predicate(value):
        try:
            return float(value) >= minimum
        except ValueError:
            return False
    return predicate


# (category, product_features keys, test on the value, description) of one attribute per query.
ATTRIBUTE_CHECKS = {
    50: ("attribute_not_met", ["legcolor"], lambda value: "gold" in value, "legcolor contains gold"),
    36: ("attribute_not_met", ["wheelsorcastersincluded", "wheelsincluded", "castersincluded"],
         lambda value: value not in ("no", "none"), "wheels or casters included"),
    277: ("wrong_product_type", ["numberofbarstoolsincluded", "numberofstoolsincluded"], number_at_least(2), "2 or more bar stools"),
    46: ("unexplained_labels", ["color"], lambda value: "blue" in value, "color contains blue"),
    402: ("unexplained_labels", ["style", "dsprimaryproductstyle", "dssecondaryproductstyle"],
          lambda value: "farmhouse" in value, "a style key contains farmhouse"),
    33: ("unexplained_labels", ["slipresistant"], lambda value: value == "yes", "slipresistant = yes"),
}


def attribute_share(product_ids, keys, predicate):
    with_attribute = sum(error_analysis.has_feature_value(products_by_id.at[product_id, "product_features"], keys, predicate)
                         for product_id in product_ids)
    return f"{with_attribute}/{len(product_ids)}"


attribute_rows = {}
for query_id, (category, keys, predicate, description) in ATTRIBUTE_CHECKS.items():
    query_labels = labels_by_query[query_id]
    by_label = {label: [product_id for product_id, value in query_labels.items() if value == label] for label in ("Exact", "Partial")}
    top10_not_exact = [product_id for product_id in selected_by_query[query_id][:10] if query_labels.get(product_id) != "Exact"]
    attribute_rows[query_id] = {
        "category": category, "query": selected_per_query.at[query_id, "query"], "attribute": description,
        "exact_with_attribute": attribute_share(by_label["Exact"], keys, predicate),
        "partial_with_attribute": attribute_share(by_label["Partial"], keys, predicate),
        "top10_not_exact_with_attribute": attribute_share(top10_not_exact, keys, predicate),
    }
plots.table(pd.DataFrame.from_dict(attribute_rows, orient="index").rename_axis("query_id"),
            "Products with the attribute asked by the query in product_features, by label, validation queries",
            note="x/y: x of the y products have the attribute. exact / partial: all judged products of the query with that label. "
                 "top10_not_exact: products in the top 10 of the selected system that are not Exact (Partial, Irrelevant or no label).")

,category,query,attribute,exact_with_attribute,partial_with_attribute,top10_not_exact_with_attribute
query_id,,,,,,
50,attribute_not_met,tufted chair with gold legs,legcolor contains gold,21/28,23/363,0/4
36,attribute_not_met,laundry basket with wheels,wheels or casters included,33/37,3/119,0/2
277,wrong_product_type,barstool patio sets,2 or more bar stools,43/44,29/152,1/7
46,unexplained_labels,beach blue headboard,color contains blue,4/5,31/97,7/8
402,unexplained_labels,modern farmhouse lighting semi flush mount,a style key contains farmhouse,0/7,18/219,7/9
33,unexplained_labels,non slip shower floor tile,slipresistant = yes,107/125,118/260,5/6


**Resultados**
- **Nas queries com causa visível, o atributo separa bem os `label`:**
  - "tufted chair with gold legs": 21 dos 28 Exact têm `legcolor` gold, contra 23 dos 363 Partial. Nenhum dos 4 produtos do top 10 que não são Exact tem.
  - "laundry basket with wheels": 33 dos 37 Exact têm rodinhas, contra 3 dos 119 Partial. Nenhum dos 2 do top 10 que não são Exact tem.
  - "barstool patio sets": 43 dos 44 Exact vêm com 2 ou mais banquetas, contra 29 dos 152 Partial. Só 1 dos 7 do top 10 que não são Exact vem. Os outros são banquetas avulsas, outro tipo de produto em relação a um "set".
- **Nas queries de `unexplained_labels`, os produtos mal colocados também têm o atributo:**
  - "beach blue headboard": 7 dos 8 produtos do top 10 que não são Exact têm `color` blue.
  - "modern farmhouse lighting semi flush mount": 7 dos 9 têm farmhouse numa key de estilo. Nenhum dos 7 Exact tem.
  - "non slip shower floor tile": 5 dos 6 têm `slipresistant = yes`.

**Impacto (nossa interpretação)**
- Em `attribute_not_met`, o sistema tem a informação nos fields e não a usa o suficiente. Uma feature do reranker que compare os atributos da query com os key:value pairs (cor de cada parte, rodinhas, quantidade) é um próximo passo concreto.
- Em `unexplained_labels`, essa feature não ajudaria: o atributo já está nos produtos mal colocados.

**Limitações**
- São 6 queries e uma regra por query, escrita pelo assistente. Para `color`, a medida é fraca: a key lista as cores de todas as variantes do produto.
- A regra pode deixar passar o atributo quando ele está só em `product_description` ou com outra key.

### 5.3 · Um exemplo de `label` sem explicação nos fields

**O que é analisado:** dois produtos da query 46, "beach blue headboard": um Partial que o sistema põe em 1º lugar e um Exact que o sistema põe em 14º.

**Por que:** para mostrar, com todos os fields, o que significa a categoria `unexplained_labels`.

**Como:** comparamos os fields dos dois produtos: `product_name`, `product_class`, a key `color`, `product_description`, `category_hierarchy` e todos os key:value pairs de `product_features`.

In [13]:
PAIR_QUERY_ID, PARTIAL_PRODUCT_ID, EXACT_PRODUCT_ID = 46, 11815, 11816
pair_rows = {}
for product_id in (PARTIAL_PRODUCT_ID, EXACT_PRODUCT_ID):
    features = set(data.parse_product_features(products_by_id.at[product_id, "product_features"]))
    pair_rows[product_id] = {"label": labels_by_query[PAIR_QUERY_ID][product_id],
                             "rank": selected_by_query[PAIR_QUERY_ID].index(product_id) + 1,
                             **products_by_id.loc[product_id, ["product_name", "product_class"]].to_dict(),
                             "color": "; ".join(value for key, value in features if key == "color")}
display(plots.table(pd.DataFrame.from_dict(pair_rows, orient="index").rename_axis("product_id"),
                    f"Two judged products of query {PAIR_QUERY_ID} ({selected_per_query.at[PAIR_QUERY_ID, 'query']!r})",
                    note="rank: position in the list of the selected system."))
partial_features = set(data.parse_product_features(products_by_id.at[PARTIAL_PRODUCT_ID, "product_features"]))
exact_features = set(data.parse_product_features(products_by_id.at[EXACT_PRODUCT_ID, "product_features"]))
print("same product_description:", products_by_id.at[PARTIAL_PRODUCT_ID, "product_description"] == products_by_id.at[EXACT_PRODUCT_ID, "product_description"])
print("same category_hierarchy:", products_by_id.at[PARTIAL_PRODUCT_ID, "category_hierarchy"] == products_by_id.at[EXACT_PRODUCT_ID, "category_hierarchy"])
print("key:value pairs of", PARTIAL_PRODUCT_ID, ":", len(partial_features), "| of", EXACT_PRODUCT_ID, ":", len(exact_features))
print("only in", PARTIAL_PRODUCT_ID, ":", sorted(partial_features - exact_features), "| only in", EXACT_PRODUCT_ID, ":", sorted(exact_features - partial_features))

,label,rank,product_name,product_class,color
product_id,,,,,
11815,Partial,1,beach upholstered panel headboard,Headboards,blue
11816,Exact,14,nautical upholstered panel headboard,Headboards,blue


same product_description: True
same category_hierarchy: True
key:value pairs of 11815 : 66 | of 11816 : 65
only in 11815 : [('bottomofheadboardtofloor', '65.6')] | only in 11816 : []


**Resultados**
- **Os dois produtos têm a mesma `product_description`, o mesmo `category_hierarchy`, o mesmo `product_class` (Headboards) e `color` blue.**
- **Os key:value pairs são quase os mesmos:** o Partial (11815) tem 66 e o Exact (11816) tem 65. A única diferença é `bottomofheadboardtofloor = 65.6`, que só o Partial tem.
- **O nome do Partial tem "beach",** a palavra da query; o do Exact tem "nautical".

**Impacto (nossa interpretação)**
- Pelos fields, o Partial atende à query tanto quanto o Exact. Nenhum sistema que use só os fields do dataset tem como separar os dois. Não encontramos o que explica a diferença, e isso é uma limitação do dataset.
- A categoria `unexplained_labels` responde por 21% da perda (Etapa 5.5). Ela deve ser reportada como limitação do dataset, não como um erro a corrigir no sistema.

**Limitações**
- É um par. A Etapa 4 mostra que pares assim são minoria: 5,6% dos pares quase idênticos têm `label` diferentes.

### 5.4 · Nomes próprios: o nome está nos fields?

**O que é analisado:** as queries de `proper_name`, que dependem de um nome de marca, coleção ou personagem.

**Por que:** se o nome está nos fields dos Exact, o sistema pode usá-lo; se não está, nenhum sistema consegue ligar o nome aos produtos certos.

**Como:** procuramos o nome, em minúsculas, no texto dos cinco fields de cada produto. Contamos em quantos produtos do catalog ele aparece e em quantos Exact da query.

In [14]:
PROPER_NAMES = {223: "wisdom stone", 481: "emma", 137: "orren ellis", 233: "kelly clarkson", 458: "bubble guppies"}
lowercase_texts = product_texts.str.lower()
name_rows = {}
for query_id, name in PROPER_NAMES.items():
    exact_ids = [product_id for product_id, label in labels_by_query[query_id].items() if label == "Exact"]
    name_rows[query_id] = {"query": selected_per_query.at[query_id, "query"], "name": name,
                           "catalog_products_with_name": int(lowercase_texts.str.contains(name, regex=False).sum()),
                           "exact_with_name": f"{int(lowercase_texts.loc[exact_ids].str.contains(name, regex=False).sum())}/{len(exact_ids)}"}
plots.table(pd.DataFrame.from_dict(name_rows, orient="index").rename_axis("query_id"),
            "Products whose text fields contain the proper name of the query, validation queries",
            note="Text fields: product_name, product_class, category_hierarchy, product_description and product_features, lowercased. "
                 "exact_with_name: x of the y Exact products of the query contain the name.")

,query,name,catalog_products_with_name,exact_with_name
query_id,,,,
223,wisdom stone river 3-3/4,wisdom stone,2,0/1
481,emma headboard,emma,17,0/1
137,orren ellis l shaped desk,orren ellis,20,1/80
233,kelly clarkson light fixtures,kelly clarkson,211,72/81
458,bubble guppies chair,bubble guppies,4,0/0


**Resultados**
- **"kelly clarkson" está em 72 dos 81 Exact** (em `product_description`) e em 211 produtos do catalog.
- **"orren ellis" está em 1 dos 80 Exact,** embora apareça em 20 produtos do catalog.
- **"wisdom stone" e "emma" não estão no único Exact das suas queries.** "bubble guppies" está em 4 produtos do catalog, e a query não tem nenhum Exact.

**Impacto (nossa interpretação)**
- Os nomes próprios formam dois casos diferentes. Quando o nome está nos fields ("kelly clarkson"), a perda é de ranking, e uma feature de match do nome em `product_description` poderia ajudar. Quando o nome não está nos Exact ("orren ellis"), os fields não ligam o nome aos produtos certos, e isso é uma limitação do dataset.

**Limitações**
- A busca é por texto exato do nome; variações de grafia não são contadas.

### 5.5 · Quanto cada categoria pesa na perda total

**Como:** a perda de uma query é 1 − NDCG@10. Somamos a perda das queries de cada categoria e dividimos pela perda somada das 97 queries do validation set. As queries com NDCG@10 ≥ 0,9 entram como "ok", porque também perdem um pouco.

In [15]:
selected_per_query["loss"] = 1 - selected_per_query["ndcg"]
category_summary = selected_per_query.groupby("category").agg(
    queries=("ndcg", "size"), mean_ndcg=("ndcg", "mean"), mean_oracle=("oracle", "mean"),
    mean_condensed_ndcg=("condensed_ndcg", "mean"), loss_sum=("loss", "sum"))
category_summary["share_of_total_loss_pct"] = category_summary["loss_sum"] / selected_per_query["loss"].sum() * 100
category_summary = category_summary.sort_values("share_of_total_loss_pct", ascending=False)
display(plots.table(category_summary[["mean_ndcg", "mean_oracle", "mean_condensed_ndcg"]],
                    "Mean NDCG@10 of the selected system by error category, validation set", precision=3,
                    note="oracle: best NDCG@10 that an ordering of the top 100 candidates can reach. condensed: products without label removed."))
plots.table(category_summary[["queries", "share_of_total_loss_pct"]], "Queries and share of the total NDCG@10 loss by error category, validation set",
            note="share_of_total_loss_pct: sum of (1 - NDCG@10) over the queries of the category divided by the sum over all 97 "
                 "validation queries.")

,mean_ndcg,mean_oracle,mean_condensed_ndcg
category,,,
unexplained_labels,0.695,0.949,0.727
wrong_product_type,0.594,0.964,0.685
typo,0.171,0.407,0.388
pooling_bias,0.431,0.750,0.750
vague_query,0.250,0.588,0.708
attribute_not_met,0.685,0.903,0.728
proper_name,0.628,0.892,0.650
compound_query,0.518,0.891,0.518
ok (NDCG@10 >= 0.9),0.981,1.000,0.983


,queries,share_of_total_loss_pct
category,,
unexplained_labels,15,21.1
wrong_product_type,11,20.6
typo,3,11.5
pooling_bias,4,10.5
vague_query,3,10.4
attribute_not_met,6,8.7
proper_name,5,8.6
compound_query,2,4.4
ok (NDCG@10 >= 0.9),48,4.2


**Resultados**
- **`unexplained_labels` (15 queries) e `wrong_product_type` (11) respondem por cerca de 21% da perda cada.**
- **Depois vêm `typo` (11,5%), `pooling_bias` (10,5%), `vague_query` (10,4%), `attribute_not_met` (8,7%), `proper_name` (8,6%) e `compound_query` (4,4%).** As 48 queries "ok" somam 4,2%.
- **Em todas as categorias, exceto `typo`, `vague_query` e `pooling_bias`, o oracle médio é ≥ 0,89:** os produtos bons estão nos candidates, e a perda é de ranking.

**Impacto (nossa interpretação)**

| Categoria | Parte da perda | O que fazer |
|---|---|---|
| `wrong_product_type` | 20,6% | Feature do reranker que compare o tipo de produto pedido na query com `product_class` |
| `attribute_not_met` | 8,7% | Features de match de atributo com os key:value pairs de `product_features` (cor de cada parte, rodinhas, quantidade) |
| `typo` | 11,5% | Corretor ortográfico; o ganho não pode ser medido com estes `label` (Etapa 3) |
| `proper_name` | 8,6% | Match do nome em `product_description` quando ele está lá; limitação do dataset quando não está |
| `compound_query` / `vague_query` | 14,8% | Não há uma ação simples; o relatório só os descreve |
| `pooling_bias` | 10,5% | Nenhuma no sistema; reportar o condensed NDCG@10 ao lado do NDCG@10 |
| `unexplained_labels` | 21,1% | Nenhuma no sistema; reportar como limitação do dataset |

- `wrong_product_type` e `attribute_not_met` juntas somam 29% da perda e têm uma causa visível nos fields. São o próximo passo mais claro para o reranker.

**Limitações**
- A parte da perda de cada categoria depende da categoria dada a cada query, que vem de uma única leitura do assistente de IA (Etapa 5.1).
- Não medimos quanto cada ação recuperaria. A tabela diz onde está a perda, não quanto dela uma mudança no sistema eliminaria.

## Etapa 6 · O sistema põe os produtos mais populares no topo?

**O que é analisado:** se, na lista do sistema escolhido, os produtos com mais ratings (`rating_count`) ficam mais perto do topo do que os outros candidates (Etapas 6.1 e 6.2), e se os erros de ordem se concentram em produtos populares ou sem rating (Etapa 6.3).

**Por que:** o reranker recebe quatro features de rating: `log_rating_count`, `log_review_count`, `average_rating` e `no_rating`. A ablation do notebook 04 (Etapa 5.3.1) mostrou que tirar essas features não muda o NDCG@10 de forma distinguível de 0 (+0,004 [−0,006; 0,015]). Mas a ablation não diz se o sistema favorece produtos populares: ele pode empurrá-los para o topo sem mudar o NDCG@10, se esses produtos não têm `label` ou têm o mesmo `label` dos que saíram. No `docs/EVALUATION.md`, a proposta de conferir se os erros se concentram em produtos muito populares ou sem rating ficou sem fazer. Se o sistema favorecer produtos populares, um produto novo, ainda sem ratings, teria menos chance de aparecer no topo.

**Como:**
1. Para cada produto, `log_rating_count` = log(1 + `rating_count`), com 0 para quem não tem rating, e `no_rating` = 1 quando o produto não tem rating (`error_analysis.product_popularity`, mesma definição das features do reranker).
2. Para cada uma das 97 queries do validation set, olhamos duas ordens dos mesmos 100 candidates: a da hybrid search (antes do reranker) e a do sistema escolhido (depois do reranker).
3. Em cada ordem, medimos (`error_analysis.popularity_profile`):
   - a mediana de `log_rating_count` no top 10 e nos 100 candidates;
   - a fração de produtos sem rating no top 10 e nos 100 candidates;
   - a Spearman correlation entre a posição (1 = topo) e `log_rating_count` nos 100 candidates. Ela vai de −1 a 1; **negativa quando os produtos mais populares ficam mais perto do topo**, 0 quando a posição não acompanha a popularidade.
4. As diferenças (top 10 − 100 candidates; sistema escolhido − hybrid search) são calculadas por query, e a tabela mostra a média entre as queries com o intervalo de 95% do bootstrap (2.000 reamostragens das queries, seed 0).

**Exemplo:** se, numa query, o top 10 tem mediana de `log_rating_count` de 4,0 (cerca de 54 ratings) e os 100 candidates têm 2,5 (cerca de 11 ratings), a diferença top 10 − candidates é +1,5: o topo é mais popular que os candidates em geral.


### 6.1 · O topo é mais popular que os candidates?

A primeira tabela mostra, para cada ordem, a média entre as 97 queries de cada medida. A segunda mostra as diferenças por query, com o intervalo de 95%: se o intervalo não contém 0, a diferença é estatisticamente significativa.


In [16]:
popularity = error_analysis.product_popularity(products)
popularity_rows = []
for query_id in validation_queries["query_id"]:
    hybrid_top100 = candidates_by_query[query_id][:ranking.CANDIDATES_K]
    selected_top100 = selected_by_query[query_id][:ranking.CANDIDATES_K]
    assert set(hybrid_top100) == set(selected_top100), "the reranker reorders the same candidates"
    for order, ranked in (("hybrid search", hybrid_top100), ("selected system", selected_top100)):
        popularity_rows.append({"query_id": query_id, "order": order,
                                **error_analysis.popularity_profile(ranked, popularity)})
popularity_per_query = pd.DataFrame(popularity_rows).pivot(index="query_id", columns="order")

POPULARITY_MEASURES = ["median_log_rating_count_top", "median_log_rating_count_pool",
                       "no_rating_share_top", "no_rating_share_pool", "spearman_position"]
display(plots.table(popularity_per_query.mean().unstack("order").loc[POPULARITY_MEASURES],
                    "Popularity of the candidates by position, mean over the validation queries", precision=3,
                    note="log_rating_count = log(1 + rating_count), 0 without rating. top: the first 10 products of the order; "
                         "pool: the 100 candidates (the same products in both orders, so the pool columns are equal). "
                         "spearman_position: Spearman correlation between position (1 = top) and log_rating_count in the 100 "
                         "candidates; negative when more popular products are closer to the top."))


def mean_with_interval(per_query_values: pd.Series) -> dict:
    values = per_query_values.dropna()
    low, high = metrics.bootstrap_mean_ci(values, seed=evaluate.BOOTSTRAP_SEED, n_resamples=evaluate.BOOTSTRAP_RESAMPLES)
    return {"mean [95% interval]": evaluate.format_ci(values.mean(), low, high), "queries": len(values)}


def per_query(measure: str, order: str) -> pd.Series:
    return popularity_per_query[(measure, order)]


popularity_comparisons = {}
for measure, label in (("median_log_rating_count", "median log_rating_count"), ("no_rating_share", "share without rating")):
    for order in ("hybrid search", "selected system"):
        popularity_comparisons[f"{label}: top 10 − 100 candidates, {order}"] = mean_with_interval(
            per_query(f"{measure}_top", order) - per_query(f"{measure}_pool", order))
    popularity_comparisons[f"{label}: top 10, selected system − hybrid search"] = mean_with_interval(
        per_query(f"{measure}_top", "selected system") - per_query(f"{measure}_top", "hybrid search"))
for order in ("hybrid search", "selected system"):
    popularity_comparisons[f"spearman_position, {order}"] = mean_with_interval(per_query("spearman_position", order))
popularity_comparisons["spearman_position: selected system − hybrid search"] = mean_with_interval(
    per_query("spearman_position", "selected system") - per_query("spearman_position", "hybrid search"))
plots.table(pd.DataFrame(popularity_comparisons).T,
            "Is the top more popular than the candidates? Per-query differences, validation set",
            note="Mean over the queries of the per-query value, with the 95% bootstrap interval "
                 f"({evaluate.BOOTSTRAP_RESAMPLES:,} resamples of the queries, seed {evaluate.BOOTSTRAP_SEED}). "
                 "Positive top 10 − candidates: the top 10 is more popular (or has more products without rating) than the 100 candidates. "
                 "spearman_position is negative when more popular products are closer to the top.")


order,hybrid search,selected system
median_log_rating_count_top,2.514,2.432
median_log_rating_count_pool,2.837,2.837
no_rating_share_top,0.226,0.242
no_rating_share_pool,0.212,0.212
spearman_position,0.045,0.047


,mean [95% interval],queries
"median log_rating_count: top 10 − 100 candidates, hybrid search",-0.323 [-0.578; -0.069],97
"median log_rating_count: top 10 − 100 candidates, selected system",-0.404 [-0.644; -0.170],97
"median log_rating_count: top 10, selected system − hybrid search",-0.082 [-0.250; 0.101],97
"share without rating: top 10 − 100 candidates, hybrid search",0.014 [-0.019; 0.047],97
"share without rating: top 10 − 100 candidates, selected system",0.030 [-0.003; 0.063],97
"share without rating: top 10, selected system − hybrid search",0.016 [-0.008; 0.040],97
"spearman_position, hybrid search",0.045 [0.016; 0.075],97
"spearman_position, selected system",0.047 [0.011; 0.083],97
spearman_position: selected system − hybrid search,0.002 [-0.023; 0.027],97


**Resultados**
- **O top 10 é um pouco menos popular que os 100 candidates, nas duas ordens.** Top 10 − candidates, mediana de `log_rating_count`: −0,323 [−0,578; −0,069] na hybrid search e −0,404 [−0,644; −0,170] no sistema escolhido. As médias entre as queries são 2,514 (top 10 da hybrid search), 2,432 (top 10 do sistema escolhido) e 2,837 (candidates); por e^x − 1, cerca de 11, 10 e 16 ratings.
- **O reranker não muda a popularidade do top 10:** sistema escolhido − hybrid search, −0,082 [−0,250; 0,101], não distinguível de 0.
- **Os produtos sem rating não ficam fora do topo:** 24,2% do top 10 do sistema escolhido e 22,6% do da hybrid search não têm rating, contra 21,2% dos candidates. As três diferenças (+0,030, +0,014 e, entre as ordens, +0,016) não são distinguíveis de 0.
- **A posição quase não acompanha a popularidade:** a Spearman correlation média entre posição e `log_rating_count` é +0,045 [0,016; 0,075] na hybrid search e +0,047 [0,011; 0,083] no sistema escolhido. O sinal positivo indica produtos um pouco mais populares mais para baixo da lista, com valores perto de 0. A diferença entre as ordens é +0,002 [−0,023; 0,027].

**Impacto (nossa interpretação)**
- **A resposta à pergunta da Etapa 6 é não:** no validation set, o sistema escolhido não põe os produtos mais populares no topo. O reranker tem quatro features de rating, mas não muda a popularidade do top 10 em relação à hybrid search, o que é coerente com a ablation do notebook 04 (sem as features de rating, o NDCG@10 não muda de forma distinguível de 0).
- **Um produto sem rating, como um produto novo, não perde espaço no topo:** a fração de produtos sem rating no top 10 não se distingue da dos candidates.

**Limitações**
- `rating_count` é só um indício de popularidade. O dataset não tem vendas nem cliques.
- A comparação é com os 100 candidates, não com o catalog. Se a hybrid search já trouxer candidates mais ou menos populares que o catalog, isso não aparece aqui.
- Os `label` medem se o produto atende à query, não se ele vende. A análise não diz se favorecer produtos populares seria melhor ou pior.
- 97 queries e nove comparações na mesma tabela, sem correção para comparações múltiplas.
- A Spearman correlation só capta relações monótonas.

**Escolhas e alternativas**

| Escolha | Status | Alternativas |
|---|---|---|
| Popularidade = log(1 + `rating_count`), com 0 para produto sem rating, a mesma definição da feature do reranker | decidida, nesta análise | `review_count`; só os produtos com rating; percentil de `rating_count` no catalog |
| Mediana do top 10 comparada com a dos 100 candidates, por query | decidida, nesta análise | média; top 5; comparar com o catalog inteiro |
| Spearman correlation entre posição e `log_rating_count` nos 100 candidates | decidida, nesta análise | ordenar só por `rating_count` e medir a sobreposição com o top 10; regressão da posição em `log_rating_count` com efeito fixo de query |


### 6.2 · Como a popularidade muda ao longo da lista

A Etapa 6.1 compara o top 10 com os 100 candidates. Aqui a lista de cada query é dividida em quatro faixas de posição (1–10, 11–25, 26–50 e 51–100), e cada faixa recebe a mediana de `log_rating_count` e a fração de produtos sem rating (`error_analysis.popularity_by_band`). A tabela mostra a média entre as 97 queries do validation set, para a ordem da hybrid search e para a do sistema escolhido.


In [17]:
band_rows = []
for query_id in validation_queries["query_id"]:
    for order, ranked in (("hybrid search", candidates_by_query[query_id]), ("selected system", selected_by_query[query_id])):
        bands = error_analysis.popularity_by_band(ranked[:ranking.CANDIDATES_K], popularity)
        band_rows.append(bands.assign(order=order, query_id=query_id).reset_index())
popularity_by_band = (pd.concat(band_rows).groupby(["band", "order"])[["median_log_rating_count", "no_rating_share"]]
                      .mean().unstack("order").reindex([f"{first}-{last}" for first, last in error_analysis.POPULARITY_BANDS]))
plots.table(popularity_by_band, "Popularity by band of positions, mean over the validation queries", precision=3,
            note="band: positions in the list (1 = top). median_log_rating_count: median of log(1 + rating_count) in the band, "
                 "0 without rating. no_rating_share: share of the band without rating. Both orders hold the same 100 candidates.")


**Resultados**
- **A faixa 1–10 é a menos popular nas duas ordens:** mediana de `log_rating_count` de 2,514 na hybrid search e 2,432 no sistema escolhido, contra 2,770 a 2,988 nas outras faixas.
- **Abaixo do top 10, a popularidade não cresce nem cai de forma regular com a posição:** 2,870, 2,770 e 2,959 na hybrid search; 2,842, 2,840 e 2,988 no sistema escolhido.
- **A fração sem rating fica entre 20,2% e 24,2% em todas as faixas;** o maior valor é o do top 10 do sistema escolhido.
- **As duas ordens ficam próximas em todas as faixas:** a maior diferença de mediana entre elas é 0,082, no top 10.

**Limitações**
- A tabela é descritiva: médias de medianas por query, sem intervalo. O teste das diferenças do top 10 está na Etapa 6.1.


### 6.3 · Os erros de ordem se concentram em produtos populares ou sem rating?

As Etapas 6.1 e 6.2 olham a popularidade da lista como um todo. Aqui olhamos os **erros**: um produto com `label` pior acima de um com `label` melhor. Duas perguntas:

1. **Um produto pior sobe mais quando é mais popular?** Um **par invertido** é um par de produtos julgados da mesma query, entre os 100 candidates, com `label` diferentes, em que o de `label` pior (Irrelevant < Partial < Exact) está acima. Separamos os pares em dois grupos: o produto pior é o **mais** popular do par, ou é o **menos** popular. Se a popularidade não interfere na ordem, a taxa de inversão é a mesma nos dois grupos. Se o produto pior é invertido mais vezes quando é o mais popular, a popularidade está causando erros.
2. **Um Exact sem rating fica mais para baixo que um Exact com rating?** Entre os Exact de uma query que estão nos 100 candidates, comparamos a posição média dos sem rating com a dos com rating, só nas queries que têm os dois. Um valor positivo quer dizer que os Exact sem rating ficam mais para baixo.

**Por que comparar taxas, e não contar inversões:** no notebook 04 (Etapa 2), os Irrelevant entre os candidates têm mais ratings que os Exact (mediana de `log_rating_count` de 3,22 contra 2,71). Então, mesmo sem nenhum efeito da popularidade, a maioria dos pares invertidos teria o produto mais popular em cima. Comparar a taxa de inversão dos dois grupos tira esse efeito.

**Como:** as duas perguntas são medidas na ordem da hybrid search, que não usa rating, e na do sistema escolhido, que usa. Os pares são os mesmos nas duas ordens. Os pares com o mesmo `log_rating_count`, como dois produtos sem rating, ficam de fora. Na primeira pergunta, as taxas somam os pares de todas as queries, e o intervalo de 95% vem de 2.000 reamostragens das queries (`error_analysis.bootstrap_pooled`); na segunda, a diferença é calculada por query e a tabela mostra a média entre as queries.

**Exemplo:** na query "gray velvet sofa", um sofá de linho (Partial) com 2.000 ratings acima de um sofá de veludo (Exact) com 4 ratings é um par invertido em que o produto pior é o mais popular.


In [18]:
inversion_rows, exact_rows = [], []
for query_id in validation_queries["query_id"]:
    query_labels = labels_by_query.get(query_id, {})
    for order, ranked in (("hybrid search", candidates_by_query[query_id]), ("selected system", selected_by_query[query_id])):
        pairs = error_analysis.judged_pair_inversions(ranked, query_labels, popularity, pool_k=ranking.CANDIDATES_K)
        inversion_rows.append({"query_id": query_id, "order": order, **error_analysis.inversion_counts(pairs)})
        exact_rows.append({"query_id": query_id, "order": order,
                           **error_analysis.exact_positions_by_rating(ranked, query_labels, popularity, pool_k=ranking.CANDIDATES_K)})
inversion_per_query = pd.DataFrame(inversion_rows).pivot(index="query_id", columns="order")
exact_per_query = pd.DataFrame(exact_rows).pivot(index="query_id", columns="order")

ORDERS = ("hybrid search", "selected system")
pair_counts = inversion_per_query.sum().unstack("order")
print(f"judged pairs with different labels and different log_rating_count: "
      f"{int(pair_counts.loc['pairs_worse_more_popular', 'hybrid search'] + pair_counts.loc['pairs_worse_less_popular', 'hybrid search']):,} "
      f"({int(pair_counts.loc['pairs_worse_more_popular', 'hybrid search']):,} with the worse product more popular), "
      f"the same in both orders")


def inversion_rate(sums: pd.Series, order: str, group: str) -> float:
    return sums[(f"inverted_worse_{group}", order)] / sums[(f"pairs_worse_{group}", order)]


def inversion_gap(sums: pd.Series, order: str) -> float:
    return inversion_rate(sums, order, "more_popular") - inversion_rate(sums, order, "less_popular")


inversion_table = {}
for order in ORDERS:
    gap = error_analysis.bootstrap_pooled(inversion_per_query, lambda sums: inversion_gap(sums, order),
                                          seed=evaluate.BOOTSTRAP_SEED, n_resamples=evaluate.BOOTSTRAP_RESAMPLES)
    inversion_table[order] = {
        "inversion rate, worse product more popular": f"{inversion_rate(inversion_per_query.sum(), order, 'more_popular'):.3f}",
        "inversion rate, worse product less popular": f"{inversion_rate(inversion_per_query.sum(), order, 'less_popular'):.3f}",
        "gap [95% interval]": evaluate.format_ci(*gap),
    }
order_gap = error_analysis.bootstrap_pooled(
    inversion_per_query, lambda sums: inversion_gap(sums, "selected system") - inversion_gap(sums, "hybrid search"),
    seed=evaluate.BOOTSTRAP_SEED, n_resamples=evaluate.BOOTSTRAP_RESAMPLES)
inversion_table["selected system − hybrid search"] = {"inversion rate, worse product more popular": "",
                                                      "inversion rate, worse product less popular": "",
                                                      "gap [95% interval]": evaluate.format_ci(*order_gap)}
plots.table(pd.DataFrame(inversion_table).T,
            "Is a worse product placed above a better one more often when it is the more popular? Validation set",
            note="Pairs of judged products of the same query, among the 100 candidates, with different labels and different "
                 "log_rating_count. Inversion rate: share of pairs where the product with the worse label is above. "
                 "gap: rate when the worse product is the more popular − rate when it is the less popular; positive when popularity "
                 "pushes worse products up. Rates pooled over the queries; interval from "
                 f"{evaluate.BOOTSTRAP_RESAMPLES:,} resamples of the queries (seed {evaluate.BOOTSTRAP_SEED}).")


judged pairs with different labels and different log_rating_count: 67,762 (33,705 with the worse product more popular), the same in both orders


,"inversion rate, worse product more popular","inversion rate, worse product less popular",gap [95% interval]
hybrid search,0.269,0.294,-0.025 [-0.072; 0.029]
selected system,0.271,0.314,-0.043 [-0.094; 0.016]
selected system − hybrid search,,,-0.018 [-0.052; 0.016]


**Resultados**
- **São 67.762 pares** de produtos julgados com `label` e `log_rating_count` diferentes; em 33.705 deles, o produto pior é o mais popular.
- **O produto pior não é invertido mais vezes quando é o mais popular.** Na hybrid search, a taxa de inversão é 0,269 quando o pior é o mais popular e 0,294 quando é o menos popular: diferença de −0,025 [−0,072; 0,029]. No sistema escolhido, 0,271 e 0,314: −0,043 [−0,094; 0,016]. Os dois intervalos contêm 0.
- **O reranker não muda essa diferença:** sistema escolhido − hybrid search, −0,018 [−0,052; 0,016].

**Impacto (nossa interpretação)**
- A popularidade não está causando erros de ordem: um produto pior não sobe por ter mais ratings, nem antes nem depois do reranker. O sinal das diferenças é até o oposto (o pior é invertido um pouco menos quando é o mais popular), mas não se distingue de 0.

**Limitações**
- Os pares abaixo do top 10 não mudam o NDCG@10, mas entram na conta com o mesmo peso que os do topo.
- Um par só diz qual produto ficou acima. Se a popularidade estiver correlacionada com outra feature (por exemplo, produtos populares com `product_description` mais longa), o efeito das duas não se separa.
- Pares de produtos sem rating dos dois lados ficam de fora, porque têm o mesmo `log_rating_count`.


In [19]:
exact_sums = exact_per_query.sum().unstack("order")
display(plots.table(pd.DataFrame({
    order: {
        "Exact with rating in the top 10 (% of those in the candidates)":
            exact_sums.loc["exact_rated_top", order] / exact_sums.loc["exact_rated", order] * 100,
        "Exact without rating in the top 10 (% of those in the candidates)":
            exact_sums.loc["exact_no_rating_top", order] / exact_sums.loc["exact_no_rating", order] * 100,
    } for order in ORDERS}), "Share of the Exact products among the candidates that reach the top 10, validation set",
    precision=1, note="Pooled over the queries: queries with many Exact weigh more. The per-query comparison is in the next table."))
print(f"Exact among the candidates: {int(exact_sums.loc['exact_rated', 'hybrid search']):,} with rating, "
      f"{int(exact_sums.loc['exact_no_rating', 'hybrid search']):,} without rating")

position_gap = {order: exact_per_query[("mean_position_no_rating", order)] - exact_per_query[("mean_position_rated", order)]
                for order in ORDERS}
position_table = {f"mean position, Exact without rating − with rating, {order}": mean_with_interval(position_gap[order])
                  for order in ORDERS}
position_table["the same, selected system − hybrid search"] = mean_with_interval(
    position_gap["selected system"] - position_gap["hybrid search"])
plots.table(pd.DataFrame(position_table).T,
            "Do Exact products without rating sit lower than Exact products with rating? Validation set",
            note="Per query, among the Exact products in the 100 candidates: mean position (1 = top) of those without rating − "
                 "mean position of those with rating; positive when the Exact without rating sit lower. Only queries with Exact of "
                 "both kinds among the candidates. Mean over the queries with the 95% bootstrap interval "
                 f"({evaluate.BOOTSTRAP_RESAMPLES:,} resamples, seed {evaluate.BOOTSTRAP_SEED}).")


,hybrid search,selected system
Exact with rating in the top 10 (% of those in the candidates),18.8,18.4
Exact without rating in the top 10 (% of those in the candidates),19.9,21.9


Exact among the candidates: 1,703 with rating, 402 without rating


,mean [95% interval],queries
"mean position, Exact without rating − with rating, hybrid search",2.105 [-2.189; 6.706],50
"mean position, Exact without rating − with rating, selected system",1.110 [-2.785; 5.310],50
"the same, selected system − hybrid search",-0.995 [-4.086; 1.941],50


**Resultados**
- **Há 1.703 Exact com rating e 402 sem rating** entre os candidates das 97 queries.
- **Os Exact sem rating chegam ao top 10 tanto quanto os com rating:** 21,9% contra 18,4% no sistema escolhido, e 19,9% contra 18,8% na hybrid search (somando as queries).
- **Na mesma query, os Exact sem rating não ficam mais para baixo de forma distinguível:** posição média sem rating − com rating de +2,105 [−2,189; 6,706] na hybrid search e +1,110 [−2,785; 5,310] no sistema escolhido, nas 50 queries que têm os dois tipos de Exact. O reranker não muda a diferença: −0,995 [−4,086; 1,941].

**Impacto (nossa interpretação)**
- **O sistema não penaliza produtos sem rating,** como um produto novo: quando ele é Exact, chega ao topo na mesma medida que um Exact com rating.
- Com a Etapa 6.3, a pergunta da proposta do `docs/EVALUATION.md` fica respondida no validation set: os erros de ordem não se concentram em produtos populares nem em produtos sem rating.

**Limitações**
- Só 50 queries têm Exact com e sem rating entre os candidates, e os intervalos são largos (cerca de ±4 posições). Um efeito pequeno não seria detectado.
- A comparação somando as queries dá mais peso às queries com muitos Exact; por isso a comparação por query é a principal.

**Escolhas e alternativas**

| Escolha | Status | Alternativas |
|---|---|---|
| Comparar a taxa de inversão quando o produto pior é o mais ou o menos popular | decidida, nesta análise | contar só as inversões com o produto popular em cima (enviesado, porque os Irrelevant têm mais ratings); regressão da inversão na diferença de `log_rating_count`, com efeito fixo de query |
| Todos os pares entre os 100 candidates | decidida, nesta análise | só os pares com pelo menos um produto no top 10, os que mudam o NDCG@10 (o conjunto de pares mudaria entre as ordens) |
| Posição média dos Exact com e sem rating, por query | decidida, nesta análise | fração de cada grupo no top 10, por query (instável com poucos Exact); incluir os Partial |


## Etapa 7 · Síntese: o que foi encontrado e o impacto

| Análise | Por que | O que foi encontrado | Impacto |
|---|---|---|---|
| **1 · Retrieval ou ranking** | Saber qual etapa melhorar | O oracle chega a 0,933, e o sistema a 0,776: o retrieval perde 0,067 e o ranking 0,156. A perda está no ranking em 39 das 49 queries com NDCG@10 < 0,9 | A próxima melhoria deve ir para o reranker, não para o retrieval |
| **2 · Produtos sem `label`** | A métrica os conta como Irrelevant; saber se isso é justo | Só 0,9% dos produtos julgados da mesma classe da query são Irrelevant. Sem os produtos sem `label`, o NDCG@10 sobe de 0,776 para 0,831 | 0,776 é provavelmente um piso; o relatório mostra os dois números |
| **3 · Typos** | Saber se um corretor ortográfico vale a pena | A query corrigida traz produtos que não foram julgados, e o NDCG@10 pode até cair | Corretor é promissor, mas o ganho não pode ser medido com estes `label` |
| **4 · Produtos quase idênticos** | Saber se os `label` são confiáveis | 5,6% dos pares quase idênticos têm `label` diferentes, contra 34,4% de todos os pares; nenhum é Exact × Irrelevant | Os `label` são consistentes na maior parte dos casos; as métricas são confiáveis |
| **5 · Categorias de erro** | Transformar "o ranking perde" em causas e ações | Tipo de produto errado (21%) e atributo não atendido (9%) têm causa visível nos fields | Próximos passos: features de tipo de produto e de atributo no reranker |
| **5.3 · `label` sem explicação** | Separar o que o sistema pode corrigir do que é limitação do dataset | Em 15 queries (21% da perda), nenhum field explica por que os produtos mal colocados são Partial e não Exact | Reportar como limitação do dataset, não como erro do sistema |
| **6 · Popularidade** | Saber se o sistema favorece produtos populares, o que a ablation das features de rating não mostra | O top 10 tem um pouco menos ratings que os candidates (−0,404 [−0,644; −0,170]); o reranker não muda isso (−0,082 [−0,250; 0,101]); a fração sem rating no top 10 não se distingue da dos candidates. Um produto pior não é invertido mais vezes quando é o mais popular (−0,043 [−0,094; 0,016]), e os Exact sem rating não ficam mais para baixo | O sistema não favorece produtos populares nem penaliza produtos sem rating; as features de rating podem sair numa versão futura, a medir no validation set |
